# Multi-Agent Systems for Production AI Engineering

## What you will build

You will build the code that audits a pull request before an enterprise team merges it. Three
reviewers check the change, one each for security, performance and style. Each of them works alone
on its own copy of the change. A final stage merges their findings into one report, and a
small request that touches only one concern goes straight to the one reviewer it needs.

Without that structure, the audit goes wrong in ways that look like success. The diagram shows the
three mistakes this course stops. One review reads the notes the other reviews left behind, a review
that never ran is reported as clean, and a one-line fix pays for the whole audit.

![What you will build](images/audit-overview.svg)

## Step 0: Set up the client and the model

Every call in this notebook goes through the repository's own client. Without an API key it replays
responses recorded from real runs, so you can follow the whole course for free, and with a key it
calls the model live.

In [1]:
import json
import time
from concurrent.futures import ThreadPoolExecutor

from vault import get_client, load_env, model_for

load_env()
client = get_client("02-multi-agent-orchestration/01-build-a-code-audit-pipeline")
MODEL = model_for("default")

print(f"Client ready. Every request in this notebook uses {MODEL}.")

Client ready. Every request in this notebook uses google/gemini-2.5-flash-lite.


## Step 1: Create the pull request under audit and its answer key

Every step in this course audits the same pull request, so we write it down first. It adds an
endpoint that exports one customer's invoices, and each line carries its number so that a reviewer
can point at the exact line it means.

In [2]:
PR_ID, PR_TITLE = "PR-4172", "Add an invoice export endpoint"
PR_LINES = [
    "import os",
    "import json",
    "from flask import request",
    "from billing.db import get_connection",
    "",
    'EXPORT_API_KEY = "exp-live-7f3a9c21d4"',
    "",
    '@app.route("/invoices/export")',
    "def exportInvoices():",
    '    customer = request.args.get("customer")',
    "    conn = get_connection()",
    '    invoices = conn.execute(f"SELECT * FROM invoices WHERE customer = \'{customer}\'").fetchall()',
    "    result = []",
    "    for invoice in invoices:",
    '        lines = conn.execute("SELECT * FROM invoice_lines WHERE invoice_id = ?",',
    '                             (invoice["id"],)).fetchall()',
    '        result.append({"invoice": dict(invoice), "lines": [dict(l) for l in lines]})',
    "    return json.dumps(result)",
]
PR_DIFF = "\n".join(f"{number:>2} | {line}" for number, line in enumerate(PR_LINES, 1))
print(PR_DIFF)

 1 | import os
 2 | import json
 3 | from flask import request
 4 | from billing.db import get_connection
 5 | 
 6 | EXPORT_API_KEY = "exp-live-7f3a9c21d4"
 7 | 
 8 | @app.route("/invoices/export")
 9 | def exportInvoices():
10 |     customer = request.args.get("customer")
11 |     conn = get_connection()
12 |     invoices = conn.execute(f"SELECT * FROM invoices WHERE customer = '{customer}'").fetchall()
13 |     result = []
14 |     for invoice in invoices:
15 |         lines = conn.execute("SELECT * FROM invoice_lines WHERE invoice_id = ?",
16 |                              (invoice["id"],)).fetchall()
17 |         result.append({"invoice": dict(invoice), "lines": [dict(l) for l in lines]})
18 |     return json.dumps(result)


We planted six problems in that change, two in each lane, so every audit can be marked against the
same answer key. A **lane** is one area of review with its own checklist, and this course has three
of them: security, performance and style.

In [3]:
LANES = ["security", "performance", "style"]
KNOWN_ISSUES = {
    "credential written into the code": ("security", {6}),
    "SQL built from request input": ("security", {12}),
    "read with no LIMIT": ("performance", {12}),
    "query inside a loop": ("performance", {14, 15, 16}),
    "unused import": ("style", {1}),
    "function name breaks PEP 8": ("style", {9}),
}


def list_issues_found(findings):
    """The known issues that at least one finding points at, by lane and line."""
    return [name for name, (lane, lines) in KNOWN_ISSUES.items()
            if any(f.get("lane") == lane and f.get("line") in lines for f in findings)]


print(f"{len(KNOWN_ISSUES)} known issues across {len(LANES)} lanes")

6 known issues across 3 lanes


## Step 2: Audit the pull request with one reviewer call

Before building several workers, we measure what a single call can do, because a pipeline has to
beat that number to be worth its extra calls. The one reviewer gets all three checklists in its
prompt and is asked to reply with its findings as JSON.

In [4]:
CHECKLISTS = {
    "security": "SQL built from request input, and credentials written into the code",
    "performance": "a database query inside a loop, and a read with no LIMIT",
    "style": "names that break PEP 8, and imports that are never used",
}
FINDINGS_SHAPE = ('Reply with JSON only: {"findings": [{"line": int, '
                  '"lane": "security|performance|style", "issue": str}]}')
PR_MESSAGE = f"{PR_ID}: {PR_TITLE}\n\n{PR_DIFF}"


def ask_model(system_prompt, user_message, max_tokens=800):
    """Send one system prompt and one user message, and return the response."""
    return client.chat.completions.create(
        model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_message}])


print(f"The reviewer reads {len(PR_MESSAGE)} characters of pull request.")

The reviewer reads 777 characters of pull request.


The next cell sends the whole audit as one request and reads the reply as JSON, the way most first
versions do.

In [5]:
ONE_REVIEWER_PROMPT = ("You review pull requests for security, performance and style. Check "
                       + "; ".join(f"{lane}: {items}" for lane, items in CHECKLISTS.items())
                       + ". " + FINDINGS_SHAPE)

started = time.perf_counter()
reply = ask_model(ONE_REVIEWER_PROMPT, PR_MESSAGE).choices[0].message.content
one_call_seconds = time.perf_counter() - started

print(f"the reply starts with: {reply[:30]!r}")
try:
    json.loads(reply)
except json.JSONDecodeError as error:
    print(f"json.loads failed: {error}")

the reply starts with: '```json\n{\n  "findings": [\n    '
json.loads failed: Expecting value: line 1 column 1 (char 0)


The model wrapped its JSON in a markdown code fence, so a plain `json.loads` fails on the very first
character. `parse_findings` removes the fence before parsing, and every worker in this notebook reads
its reply through it.

In [6]:
def parse_findings(reply):
    """Read the findings list, even when the model wraps its JSON in a code fence."""
    body = reply.strip()
    if body.startswith("```"):
        body = body.split("```")[1].removeprefix("json").strip()
    return json.loads(body)["findings"]


one_call_found = list_issues_found(parse_findings(reply))
print(f"one call found {len(one_call_found)} of {len(KNOWN_ISSUES)} known issues "
      f"in {one_call_seconds:.1f} seconds")
for name in KNOWN_ISSUES:
    print(f"  {'found ' if name in one_call_found else 'missed'}  {name}")

one call found 5 of 6 known issues in 0.9 seconds
  found   credential written into the code
  found   SQL built from request input
  found   read with no LIMIT
  found   query inside a loop
  missed  unused import
  found   function name breaks PEP 8


One call found five of the six known issues in under a second, and it missed only the unused import
on line 1. That is the number the pipeline has to match, and on a change this small a single
reviewer is already a strong baseline.

## Step 3: Split the audit into three serialized worker tasks

We now split the audit into one task per lane, so that each worker gets a single job and a short
checklist. The part of a pipeline that does this split is the **orchestrator**, and handing each
task to its own worker is the **orchestrator-worker** pattern.

![Split the audit into three serialized worker tasks](images/audit-pipeline-step-1.svg)

Here the split is always the same three lanes, so plain code does it and no model call is needed.
Each task is written as JSON, which is **state serialization**: turning the state a worker needs
into plain text, so the task can cross to another thread, process or machine and arrive intact.

In [7]:
def plan_audit_tasks(pr_id, pr_message, lanes):
    """The orchestrator: one task per lane, each a plain dict that survives JSON."""
    return [{"pr_id": pr_id, "lane": lane, "checklist": CHECKLISTS[lane], "diff": pr_message}
            for lane in lanes]


AUDIT_TASKS = plan_audit_tasks(PR_ID, PR_MESSAGE, LANES)
packet = json.dumps(AUDIT_TASKS[0])

print(f"{len(AUDIT_TASKS)} tasks planned: {[task['lane'] for task in AUDIT_TASKS]}")
print(f"one task serialized is {len(packet)} characters with keys {list(AUDIT_TASKS[0])}")
print(f"it reads back unchanged: {json.loads(packet) == AUDIT_TASKS[0]}")

3 tasks planned: ['security', 'performance', 'style']
one task serialized is 948 characters with keys ['pr_id', 'lane', 'checklist', 'diff']
it reads back unchanged: True


## Step 4: Give each worker its own isolated context

The tempting way to run three reviews is one conversation that asks for each lane in turn. The next
cell does exactly that and prints how many prompt tokens each pass sends, where a **token** is the
unit a model reads and bills in, roughly a short word or part of one.

![Give each worker its own isolated context](images/audit-pipeline-step-2.svg)

In [8]:
shared_messages = [{"role": "system", "content": "You review pull requests. " + FINDINGS_SHAPE},
                   {"role": "user", "content": PR_MESSAGE}]
shared_findings = []

for task in AUDIT_TASKS:
    shared_messages.append({"role": "user", "content":
                            f"Now report only {task['lane']} issues. Check {task['checklist']}."})
    response = client.chat.completions.create(model=MODEL, max_tokens=800,
                                              messages=shared_messages)
    reply = response.choices[0].message.content
    shared_messages.append({"role": "assistant", "content": reply})
    reported = parse_findings(reply)
    shared_findings += reported
    print(f"{task['lane']:<12} prompt tokens {response.usage.prompt_tokens:>4}   "
          f"lines reported {sorted({f['line'] for f in reported})}")

print(f"\none shared conversation found {len(list_issues_found(shared_findings))} "
      f"of {len(KNOWN_ISSUES)} known issues")

security     prompt tokens  305   lines reported [6, 12]


performance  prompt tokens  471   lines reported [15]


style        prompt tokens  573   lines reported [8]

one shared conversation found 3 of 6 known issues


The prompt grew from 305 to 573 tokens across the three passes, because every pass carries the
change again along with every reply before it. The later passes also got worse rather than better.
The performance pass reported only line 15, and the style pass reported line 8, the route decorator,
while it missed both of the style issues we planted.

**Context isolation** means each worker sees only its own task, never the conversation or the
findings of another worker. `run_worker` builds its messages from one task and nothing else, and
returns the findings with the prompt tokens and the seconds the call took.

In [9]:
def build_worker_prompt(task):
    """The system prompt for one lane, taken from the task alone."""
    return (f"You are the {task['lane']} reviewer. Report only {task['lane']} issues. "
            f"Check {task['checklist']}. {FINDINGS_SHAPE}")


def run_worker(task):
    """One worker, one lane. Its messages are built from its task and nothing else."""
    started = time.perf_counter()
    response = ask_model(build_worker_prompt(task), task["diff"])
    return {"lane": task["lane"],
            "findings": parse_findings(response.choices[0].message.content),
            "prompt_tokens": response.usage.prompt_tokens,
            "seconds": time.perf_counter() - started}


print("run_worker builds every request from one task")

run_worker builds every request from one task


The next cell runs the three workers one after another, which also gives us the time to beat in
the next step.

In [10]:
started = time.perf_counter()
serial_results = [run_worker(task) for task in AUDIT_TASKS]
serial_seconds = time.perf_counter() - started

for result in serial_results:
    print(f"{result['lane']:<12} prompt tokens {result['prompt_tokens']:>4}   "
          f"lines reported {sorted({f['line'] for f in result['findings']})}   "
          f"{result['seconds']:.1f} s")
serial_found = list_issues_found([f for result in serial_results for f in result["findings"]])
print(f"\nisolated workers found {len(serial_found)} of {len(KNOWN_ISSUES)} known issues")
print(f"three workers one after another: {serial_seconds:.1f} seconds")

security     prompt tokens  304   lines reported [6, 12]   1.2 s
performance  prompt tokens  305   lines reported [12, 15]   0.8 s
style        prompt tokens  305   lines reported [1, 9]   0.7 s

isolated workers found 6 of 6 known issues
three workers one after another: 2.7 seconds


Every worker now sends about 305 prompt tokens, because none of them carries another worker's reply.
With the same three checklists, the isolated workers found all six known issues, including both of
the style issues that the shared conversation missed. The run took 2.7 seconds, because each worker
waited for the one before it to finish.

## Step 5: Fan the three workers out in parallel

The three workers do not depend on each other, so there is no reason to wait for one before starting
the next. To **fan out** is to start several workers at once and collect their results when all of
them have finished, and here a small thread pool does it.

In [11]:
def fan_out_workers(tasks):
    """Start every worker at once. The slowest one sets the wall clock time."""
    with ThreadPoolExecutor(max_workers=len(tasks)) as pool:
        return list(pool.map(run_worker, tasks))


started = time.perf_counter()
parallel_results = fan_out_workers(AUDIT_TASKS)
parallel_seconds = time.perf_counter() - started

for result in parallel_results:
    print(f"{result['lane']:<12} finished in {result['seconds']:.1f} s")
print(f"\none after another: {serial_seconds:.1f} s, in parallel: {parallel_seconds:.1f} s")

security     finished in 0.8 s
performance  finished in 1.1 s
style        finished in 0.7 s

one after another: 2.7 s, in parallel: 1.1 s


In parallel the audit took 1.1 seconds, which is the time of its slowest worker, instead of the 2.7
second sum of all three. When the notebook runs from saved responses, it waits as long as each
recorded call took, so these timings read the same with or without a key.

## Step 6: Merge the findings and write one report

The last stage is the **synthesizer**, which merges every worker's findings into one report for the
author of the pull request. The merge itself is plain code, so no finding can be lost in a model's
retelling, and one short model call writes the summary on top of it.

![Merge the findings and write one report](images/audit-pipeline-step-3.svg)

In [12]:
SYNTHESIZER_PROMPT = "You write the audit summary for a pull request author, in two sentences."


def merge_findings(results):
    """Collect every worker's findings under its lane, one list per lane."""
    merged = {lane: [] for lane in LANES}
    for result in results:
        merged[result["lane"]] = result["findings"]
    return merged


def write_audit_summary(merged):
    """One model call that turns the merged findings into two sentences."""
    return ask_model(SYNTHESIZER_PROMPT, json.dumps(merged), max_tokens=300) \
        .choices[0].message.content


print(f"the synthesizer merges {len(LANES)} lanes into one report")

the synthesizer merges 3 lanes into one report


Now the pipeline can be marked against the answer key, next to the one call from Step 2.

In [13]:
merged = merge_findings(parallel_results)
pipeline_found = list_issues_found([f for findings in merged.values() for f in findings])

print(f"one reviewer call : {len(one_call_found)} of {len(KNOWN_ISSUES)} known issues")
print(f"three workers     : {len(pipeline_found)} of {len(KNOWN_ISSUES)} known issues")
print(f"missed this run   : {[name for name in KNOWN_ISSUES if name not in pipeline_found]}")
print(f"\nsummary: {write_audit_summary(merged)}")

one reviewer call : 5 of 6 known issues
three workers     : 5 of 6 known issues
missed this run   : ['unused import']



summary: This pull request has a few critical security issues, including a hardcoded API key and a SQL injection vulnerability, which should be addressed immediately. Additionally, there are performance concerns regarding database queries within loops and some minor style suggestions for improved readability.


This run of the workers found five of the six issues, the same count as the one call in Step 2, and
it missed the same unused import. The sequential workers in Step 4 found all six, because model
answers vary from run to run. On a change this small, the split buys isolation, speed and a
separate result for each lane rather than more findings. The next step shows why that separate
result matters.

## Step 7: Report a worker that never ran instead of a clean lane

A worker can fail in production, because its request times out or its reply cannot be parsed. Most
first versions of fan out code log the failure and carry on, so the next cell makes the security
worker time out and runs the audit exactly that way.

![Report a worker that never ran instead of a clean lane](images/audit-pipeline-step-4.svg)

In [14]:
WORKER_OUTAGES = {"security"}   # lanes whose worker times out on the next run


def run_worker_over_flaky_network(task):
    """Run the worker, unless its lane is having an outage."""
    if task["lane"] in WORKER_OUTAGES:
        raise TimeoutError(f"the {task['lane']} worker did not reply")
    return run_worker(task)


def fan_out_and_skip_failures(tasks):
    """The usual first version: log a failed worker and carry on without it."""
    results = []
    with ThreadPoolExecutor(max_workers=len(tasks)) as pool:
        for future in [pool.submit(run_worker_over_flaky_network, task) for task in tasks]:
            try:
                results.append(future.result())
            except TimeoutError as error:
                print(f"worker failed and was skipped: {error}")
    return results

The run below merges whatever came back and asks the synthesizer for its summary.

In [15]:
outage_results = fan_out_and_skip_failures(AUDIT_TASKS)
merged = merge_findings(outage_results)

print(f"security findings in the merge: {merged['security']}")
print(f"summary: {write_audit_summary(merged)}")

worker failed and was skipped: the security worker did not reply


security findings in the merge: []


summary: The pull request contains performance issues related to executing queries within loops and queries without a LIMIT clause, which could lead to N+1 query problems and excessive memory consumption. Additionally, there are style suggestions regarding function naming conventions and string formatting for improved code readability and adherence to PEP 8 guidelines.


The merge filled the missing security lane with an empty list, which looks exactly like a lane that
ran and found nothing. The summary then describes the performance and style issues and never
mentions security, so the author has no way to tell that the hardcoded key and the SQL built from
request input were never checked.

The fix is to merge by the plan rather than by what came back. `merge_findings_by_status` lists
every lane the orchestrator planned and marks each one as done or as not run, and
`decide_merge_verdict` blocks the merge in code whenever a lane is missing, whatever the summary
says.

In [16]:
def merge_findings_by_status(tasks, results):
    """Every planned lane appears, marked done or not run, so silence never reads as clean."""
    finished = {result["lane"]: result["findings"] for result in results}
    return {task["lane"]: {"status": "done", "findings": finished[task["lane"]]}
            if task["lane"] in finished else {"status": "did not run", "findings": []}
            for task in tasks}


def decide_merge_verdict(merged):
    """Blocked if any lane never ran, then changes requested if anything was found."""
    missing = [lane for lane, entry in merged.items() if entry["status"] != "done"]
    if missing:
        return f"blocked: rerun the {', '.join(missing)} review"
    return "changes requested" if any(e["findings"] for e in merged.values()) else "approved"


print("the merge now follows the plan, not the replies")

the merge now follows the plan, not the replies


The same failed run now goes through the new merge.

In [17]:
merged = merge_findings_by_status(AUDIT_TASKS, outage_results)

for lane, entry in merged.items():
    print(f"{lane:<12} {entry['status']:<12} {len(entry['findings'])} findings")
print(f"\nverdict: {decide_merge_verdict(merged)}")
print(f"summary: {write_audit_summary(merged)}")

security     did not run  0 findings
performance  done         2 findings
style        done         2 findings

verdict: blocked: rerun the security review


summary: The audit found two performance issues related to inefficient database queries, specifically N+1 query problems and a lack of LIMIT clauses. Additionally, there are two style issues regarding function naming conventions and string formatting.


The merge now reports that security did not run, and the verdict blocks the pull request until that
review runs again. The summary still leaves security out, even though its input said the lane did
not run, which is why the verdict is decided in code rather than by the summary.

## Step 8: Route small requests to a single specialist

Most pull requests are small and touch a single concern. Sending each of them through three workers
and a synthesizer spends four calls on work that one worker could do. The **router pattern** puts
one short call in front of the pipeline, which reads the request and sends it either to the full
audit or to the one specialist it needs.

![Route small requests to a single specialist](images/audit-pipeline-step-5.svg)

In [18]:
ROUTER_PROMPT = ("Route this pull request. Reply with one word: full, security, performance "
                 "or style. Use full for new features and for anything touching several concerns.")
ROUTES = ["full", *LANES]


def read_route(reply):
    """Normalise the router's word. Anything unknown gets the full audit, never nothing."""
    route = (reply or "").strip().lower().strip(".")
    return route if route in ROUTES else "full"


def route_request(pr_title):
    """One short call on the title alone, returning a route the pipeline has a path for."""
    return read_route(ask_model(ROUTER_PROMPT, pr_title, max_tokens=10).choices[0].message.content)


print(f"routes the pipeline can take: {ROUTES}")

routes the pipeline can take: ['full', 'security', 'performance', 'style']


Five requests show the router at work, each next to the route a senior reviewer picked. The full
audit costs three workers and a synthesizer, while a routed request costs the router and one worker.

In [19]:
INCOMING_REQUESTS = [
    ("PR-4180", "Fix a typo in the export error message", "style"),
    ("PR-4181", "Rename exportInvoices to export_invoices", "style"),
    ("PR-4182", "Add a LIMIT to the invoices query", "performance"),
    ("PR-4183", "Move the export API key into an environment variable", "security"),
    ("PR-4184", "Add a refund endpoint with a new database query", "full"),
]
FULL_AUDIT_CALLS = len(LANES) + 1

routed_calls = 0
for pr_id, pr_title, expected in INCOMING_REQUESTS:
    route = route_request(pr_title)
    routed_calls += 1 + (FULL_AUDIT_CALLS if route == "full" else 1)
    print(f"{pr_id}  routed to {route:<12} expected {expected}")

print(f"\ncalls without a router: {FULL_AUDIT_CALLS * len(INCOMING_REQUESTS)}")
print(f"calls with the router : {routed_calls}")

PR-4180  routed to style        expected style


PR-4181  routed to style        expected style


PR-4182  routed to full         expected performance


PR-4183  routed to security     expected security


PR-4184  routed to full         expected full

calls without a router: 20
calls with the router : 16


The router sent four of the five requests where the senior reviewer did, and it cut the batch from
20 calls to 16. It sent the LIMIT change to the full audit instead of the performance worker, which
costs three extra calls but misses nothing. Any word the router returns that is not a known route
falls back to the full audit in the same way.

## Step 9: Test the pipeline without calling the model

Each safeguard above gets a test that runs in milliseconds with no API key, so it can run on every
commit. If someone lets a failed worker turn back into an empty lane, or adds a route with no worker
behind it, one of these tests fails.

In [20]:
def test_every_route_has_a_path():
    for route in ROUTES:
        lanes = LANES if route == "full" else [route]
        assert all(lane in CHECKLISTS for lane in lanes), f"{route} has no worker"
    assert read_route("Security.") == "security"
    assert read_route("database") == "full"


def test_missing_worker_blocks_the_merge():
    done = [{"lane": "performance", "findings": []}, {"lane": "style", "findings": []}]
    merged = merge_findings_by_status(AUDIT_TASKS, done)
    assert merged["security"]["status"] == "did not run"
    assert decide_merge_verdict(merged).startswith("blocked")

The last two tests cover the task packet and the fenced reply, because every worker depends on
both of them.

In [21]:
def test_task_packet_carries_only_its_task():
    task = plan_audit_tasks("PR-1", "1 | import os", ["style"])[0]
    assert json.loads(json.dumps(task)) == task
    assert set(task) == {"pr_id", "lane", "checklist", "diff"}


def test_fenced_reply_parses():
    fenced = '```json\n{"findings": [{"line": 1, "lane": "style", "issue": "x"}]}\n```'
    assert parse_findings(fenced)[0]["line"] == 1


for test in (test_every_route_has_a_path, test_missing_worker_blocks_the_merge,
             test_task_packet_carries_only_its_task, test_fenced_reply_parses):
    test()
    print(f"passed: {test.__name__}")

passed: test_every_route_has_a_path
passed: test_missing_worker_blocks_the_merge
passed: test_task_packet_carries_only_its_task
passed: test_fenced_reply_parses


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Orchestrator** | `plan_audit_tasks` | Splits one audit into one task per lane |
| **State serialization** | the task dict and `json.dumps` | Carries everything a worker needs as plain text that reads back unchanged |
| **Context isolation** | `run_worker` | Builds each worker's messages from its own task only |
| **Parallel fan out** | `fan_out_workers` | Runs the workers at once, so the slowest sets the time |
| **Synthesis** | `merge_findings_by_status` and `write_audit_summary` | Merges the findings in code, then writes one summary |
| **Missing lane** | `decide_merge_verdict` | Blocks the merge when a planned worker never ran |
| **Router pattern** | `route_request` and `read_route` | Sends a small request to one specialist, and anything unknown to the full audit |